[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-00-baselines-first-defaults-and-learning-curves.ipynb)

# Advanced Discussion 0 — baselines first: five models, four datasets and a learning curve

Repeated cross-validation of a majority guess, logistic regression, a random forest, gradient boosting and a small neural network, fit-time comparison, and learning curves on digits.

**Optional advanced-discussion lab** for Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Uses scikit-learn's bundled datasets; takes a few minutes on CPU.

In [ ]:
%pip install -q scikit-learn numpy

In [ ]:
import time, warnings, numpy as np
warnings.filterwarnings("ignore")
from sklearn.datasets import load_breast_cancer, load_wine, load_digits, make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
rs = np.random.RandomState(0)

def models(seed=0):
    return {"majority class": DummyClassifier(strategy="most_frequent"),
            "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
            "random forest": RandomForestClassifier(200, random_state=seed, n_jobs=-1),
            "gradient boosting": HistGradientBoostingClassifier(random_state=seed),
            "small neural net": make_pipeline(StandardScaler(), MLPClassifier((64, 64), max_iter=800, alpha=1e-2, random_state=seed)),
            "small neural net (early stopping)": make_pipeline(StandardScaler(), MLPClassifier((64, 64), max_iter=800, early_stopping=True, random_state=seed))}

## 1. baselines first: five models, repeated cross-validation, on four tabular problems

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True); X_w, y_w = load_wine(return_X_y=True); X_d, y_d = load_digits(return_X_y=True)
# a synthetic problem with irrelevant features and an interaction, where a tree might struggle less than a linear model
X_s, y_s = make_classification(n_samples=1000, n_features=30, n_informative=6, n_redundant=4, n_clusters_per_class=4, flip_y=0.03, class_sep=0.8, random_state=0)
# breast cancer with 60 pure-noise features appended
X_bcn = np.hstack([X_bc, rs.randn(len(X_bc), 60)])
tasks = {"breast cancer (569 x 30)": (X_bc, y_bc), "wine (178 x 13)": (X_w, y_w), "digits (1,797 x 64)": (X_d, y_d), "synthetic, clustered classes (1,000 x 30)": (X_s, y_s), "breast cancer + 60 noise columns": (X_bcn, y_bc)}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)
print("accuracy, 5-fold cross-validation repeated 3 times (mean +/- sd over the 15 folds)")
names = [n for n in models() if "early" not in n]
print("%-42s" % "task" + "".join("%-22s" % n for n in names))
res = {}
for tname, (X, y) in tasks.items():
    row = []
    for n, m in models().items():
        if "early" in n: continue
        s = cross_val_score(m, X, y, cv=cv); res[(tname, n)] = s.mean(); row.append("%.3f +/- %.3f      " % (s.mean(), s.std()))
    print("%-42s" % tname + "".join(row))
print("\nbest model per task: " + "; ".join("%s -> %s" % (t.split(" (")[0], max(names, key=lambda n: res[(t, n)])) for t in tasks))

## 2. fit time

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X_d, y_d, test_size=0.3, random_state=0, stratify=y_d)
print("\nfit time on digits (1,258 training rows), seconds:")
for n, m in models().items():
    if "early" in n: continue
    t0 = time.perf_counter(); m.fit(Xtr, ytr); print("  %-22s %.3f s   test accuracy %.3f" % (n, time.perf_counter() - t0, m.score(Xte, yte)))

## 3. learning curves: where does the neural net start to win? (digits, repeated draws)

In [ ]:
sizes = (30, 60, 120, 250, 500, 1000)
print("\ndigits: test accuracy vs number of training rows (mean of 10 random draws)")
print("%-34s" % "model" + "".join("%8d" % s for s in sizes))
for n in ("logistic regression", "random forest", "gradient boosting", "small neural net", "small neural net (early stopping)"):
    accs = []
    for s in sizes:
        a = []
        for seed in range(10):
            Xa, Xb, ya, yb = train_test_split(X_d, y_d, train_size=s, test_size=600, random_state=seed, stratify=y_d)
            m = models(seed)[n]; m.fit(Xa, ya); a.append(m.score(Xb, yb))
        accs.append(np.mean(a))
    print("%-34s" % n + "".join("%8.3f" % a for a in accs))

## 4. the same idea in one sentence of code: a baseline you must beat

In [ ]:
print("\nrule for the decision dossier: report the strongest simple baseline next to the neural net, with the same split and the same metric.")

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.